In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔒 Container Security & Kubernetes Security Guide

**Securing containerized applications and Kubernetes clusters from threats**

This guide covers:
- Container image scanning and signing
- Runtime security and admission controllers
- Pod security policies and network policies
- Secrets management
- RBAC in Kubernetes
- Compliance and auditing
- Vulnerability detection
- Security scanning in CI/CD
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Container Image Security

### Image Scanning in CI/CD
```yaml
# .github/workflows/container-security.yml
name: Container Security Scan

on:
  push:
    branches: [main]
    paths: ['Dockerfile', 'requirements.txt']
  pull_request:
    branches: [main]

jobs:
  scan:
    runs-on: ubuntu-latest
    
    steps:
      - uses: actions/checkout@v3
      
      # Build image
      - name: Build Docker Image
        run: docker build -t my-app:${{ github.sha }} .
      
      # Scan with Trivy
      - name: Run Trivy Vulnerability Scanner
        uses: aquasecurity/trivy-action@master
        with:
          image-ref: my-app:${{ github.sha }}
          format: sarif
          output: trivy-results.sarif
          severity: CRITICAL,HIGH
      
      # Upload to GitHub Security
      - name: Upload Trivy Results
        uses: github/codeql-action/upload-sarif@v2
        with:
          sarif_file: trivy-results.sarif
      
      # Scan with Snyk
      - name: Run Snyk Scan
        uses: snyk/actions/docker@master
        env:
          SNYK_TOKEN: ${{ secrets.SNYK_TOKEN }}
        with:
          image: my-app:${{ github.sha }}
          args: --severity-threshold=high
```

### Multi-Stage Build (Secure)
```dockerfile
# Dockerfile - Multi-stage build for reduced attack surface

# Stage 1: Build
FROM python:3.11-slim as builder

WORKDIR /build

# Install build dependencies only in builder
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# Stage 2: Runtime (minimal image)
FROM python:3.11-slim

WORKDIR /app

# Create non-root user
RUN useradd -m -u 1000 appuser

# Copy only necessary files from builder
COPY --from=builder /root/.local /home/appuser/.local
COPY app.py .

# Set environment
ENV PATH=/home/appuser/.local/bin:$PATH
ENV PYTHONUNBUFFERED=1

# Switch to non-root user
USER appuser

# Security: Make filesystem read-only
RUN chmod -R a-w /app

# Healthcheck
HEALTHCHECK --interval=30s --timeout=3s --start-period=5s --retries=3 \
    CMD python -c "import socket; socket.create_connection(('localhost', 8000), timeout=2)"

# Non-root user runs the app
CMD ["python", "app.py"]

# Metadata
LABEL security.scan="enabled"
LABEL maintainer="security@example.com"
```

### Image Signing (Cosign)
```bash
#!/bin/bash
# sign-and-push-image.sh

IMAGE="my-registry.azurecr.io/my-app:$VERSION"

# Build image
docker build -t $IMAGE .

# Push to registry
docker push $IMAGE

# Install Cosign (if not present)
# curl -Lo /usr/local/bin/cosign https://github.com/sigstore/cosign/releases/latest/download/cosign-linux-amd64
# chmod +x /usr/local/bin/cosign

# Sign image
export COSIGN_EXPERIMENTAL=1
cosign sign --key cosign.key $IMAGE

# Verify signature
cosign verify --key cosign.pub $IMAGE
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Kubernetes Security

### Pod Security Policies
```yaml
# pod-security-policy.yaml
apiVersion: policy/v1beta1
kind: PodSecurityPolicy
metadata:
  name: restricted-psp
spec:
  # Prevent privileged escalation
  privileged: false
  allowPrivilegeEscalation: false
  
  # Require read-only root filesystem
  readOnlyRootFilesystem: true
  
  # Restrict volumes
  volumes:
    - configMap
    - emptyDir
    - projected
    - secret
    - downwardAPI
    - persistentVolumeClaim
  
  # Require non-root user
  runAsUser:
    rule: MustRunAsNonRoot
  
  # Drop capabilities
  requiredDropCapabilities:
    - ALL
  
  # Allow specific capabilities only if needed
  allowedCapabilities:
    - NET_BIND_SERVICE
  
  # SELinux
  seLinux:
    rule: MustRunAs
    seLinuxOptions:
      level: "s0:c123,c456"
  
  # Host network/IPC not allowed
  hostNetwork: false
  hostIPC: false
  hostPID: false

---
# ClusterRole to bind PSP
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: restricted-psp-user
rules:
  - apiGroups: ['policy']
    resources: ['podsecuritypolicies']
    verbs: ['use']
    resourceNames:
      - restricted-psp

---
# ClusterRoleBinding
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: restricted-psp-binding
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: restricted-psp-user
subjects:
  - kind: ServiceAccount
    name: default
    namespace: default
```

### Network Policies
```yaml
# network-policy.yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: app-netpol
spec:
  podSelector:
    matchLabels:
      app: my-app
  
  policyTypes:
    - Ingress
    - Egress
  
  # Ingress: Allow only from specific namespaces/pods
  ingress:
    - from:
        - namespaceSelector:
            matchLabels:
              name: production
        - podSelector:
            matchLabels:
              role: frontend
      ports:
        - protocol: TCP
          port: 8000
  
  # Egress: Allow only to specific destinations
  egress:
    # Allow DNS
    - to:
        - namespaceSelector: {}
          podSelector:
            matchLabels:
              k8s-app: kube-dns
      ports:
        - protocol: UDP
          port: 53
    
    # Allow to database
    - to:
        - podSelector:
            matchLabels:
              app: postgres
      ports:
        - protocol: TCP
          port: 5432
    
    # Allow external HTTPS
    - to:
        - namespaceSelector: {}
      ports:
        - protocol: TCP
          port: 443
```

### RBAC (Role-Based Access Control)
```yaml
# rbac.yaml
---
# ServiceAccount for app
apiVersion: v1
kind: ServiceAccount
metadata:
  name: my-app-sa
  namespace: default

---
# Role with minimal permissions
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: my-app-role
  namespace: default
rules:
  # Only allow reading ConfigMaps and Secrets in this namespace
  - apiGroups: [""]
    resources: ["configmaps", "secrets"]
    verbs: ["get", "list"]
    resourceNames:
      - app-config
      - app-secrets
  
  # Allow reading Pods (for debugging)
  - apiGroups: [""]
    resources: ["pods", "pods/log"]
    verbs: ["get", "list"]

---
# RoleBinding
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: my-app-rolebinding
  namespace: default
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: Role
  name: my-app-role
subjects:
  - kind: ServiceAccount
    name: my-app-sa
    namespace: default

---
# Pod using ServiceAccount with restricted permissions
apiVersion: v1
kind: Pod
metadata:
  name: my-app-pod
  namespace: default
spec:
  serviceAccountName: my-app-sa
  
  containers:
    - name: app
      image: my-app:latest
      
      # Security context
      securityContext:
        runAsNonRoot: true
        runAsUser: 1000
        readOnlyRootFilesystem: true
        allowPrivilegeEscalation: false
        capabilities:
          drop:
            - ALL
      
      # Resource limits
      resources:
        requests:
          cpu: 100m
          memory: 128Mi
        limits:
          cpu: 500m
          memory: 512Mi
      
      # Probes
      livenessProbe:
        httpGet:
          path: /health
          port: 8000
        initialDelaySeconds: 10
        periodSeconds: 10
      
      readinessProbe:
        httpGet:
          path: /ready
          port: 8000
        initialDelaySeconds: 5
        periodSeconds: 5
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Secrets Management

### Sealed Secrets in Kubernetes
```bash
#!/bin/bash
# Setup Sealed Secrets

# Install Sealed Secrets controller
kubectl apply -f https://github.com/bitnami-labs/sealed-secrets/releases/download/v0.20.0/controller.yaml

# Create a secret
kubectl create secret generic app-db-secret \
  --from-literal=db_host=postgres.default.svc.cluster.local \
  --from-literal=db_password=secure_password \
  -o yaml > secret.yaml

# Seal the secret
kubeseal -f secret.yaml -w sealed-secret.yaml

# Apply sealed secret (safe to commit to git)
kubectl apply -f sealed-secret.yaml

# Verify secret was created
kubectl get secret app-db-secret
```

### Environment-based Secrets
```python
# secrets.py
import os
from typing import Optional

class SecretManager:
    """Manage secrets from environment variables"""
    
    REQUIRED_SECRETS = [
        'DB_PASSWORD',
        'API_KEY',
        'SECRET_KEY'
    ]
    
    @staticmethod
    def validate():
        """Ensure all required secrets are present"""
        missing = []
        
        for secret_name in SecretManager.REQUIRED_SECRETS:
            if not os.environ.get(secret_name):
                missing.append(secret_name)
        
        if missing:
            raise ValueError(f"Missing secrets: {', '.join(missing)}")
    
    @staticmethod
    def get(key: str, default: Optional[str] = None) -> str:
        """Get secret with timeout to prevent logging"""
        return os.environ.get(key, default)

# Usage
SecretManager.validate()
db_password = SecretManager.get('DB_PASSWORD')
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 4. Container Security Best Practices

✅ **Image Security**
- [ ] Scan images for vulnerabilities (Trivy, Snyk)
- [ ] Use minimal base images (Alpine, distroless)
- [ ] Multi-stage builds to reduce size
- [ ] Sign images with Cosign
- [ ] Regular updates for base images

✅ **Kubernetes Security**
- [ ] Pod Security Policies enforced
- [ ] Network Policies configured
- [ ] RBAC with least privilege
- [ ] Resource limits set
- [ ] Non-root user enforcement

✅ **Runtime Security**
- [ ] Image pull policies (Always)
- [ ] Registry credentials in secrets
- [ ] Admission controllers enabled
- [ ] Falco for runtime monitoring
- [ ] Audit logging enabled

✅ **Secrets**
- [ ] Never in image or logs
- [ ] Use sealed-secrets or external vault
- [ ] Rotate regularly
- [ ] Audit access logs
- [ ] Environment-based injection

✅ **Compliance**
- [ ] CIS Kubernetes Benchmarks
- [ ] OWASP Top 10 for containers
- [ ] Regular security audits
- [ ] Penetration testing
- [ ] Compliance scanning
</MySCode.Cell>
```